# Phase 3 — `player_career_stats`

We roll the atomic `player_match_stats` up to **one row per player**, summing
counting stats across every match in every competition, then deriving:

* **per-90 metrics** — `goals_p90`, `key_passes_p90`, `clearances_p90`, … the
  pace-independent inputs the rating attributes will consume.
* **accuracy rates** — pass completion, shot accuracy, clearance accuracy,
  goalkeeper save% (which needs goals conceded, derived from the match score).
* **coverage** — total minutes, matches, and the list of competitions a
  player featured in (drives the confidence stars later).

**Still no ratings.** This phase only produces clean aggregated features.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import wyscout_lib as wl

pd.set_option("display.max_columns", 90)
pd.set_option("display.width", 200)
DATA = wl.DATA

pms = pd.read_parquet(DATA / "player_match_stats.parquet")
matches = pd.read_parquet(DATA / "matches_master.parquet")
players = pd.read_parquet(DATA / "players_master.parquet")

career = wl.build_player_career_stats(pms, matches, players)
career.to_parquet(DATA / "player_career_stats.parquet")
print(f"player_career_stats: {career.shape}")
print(f"rated players (>= {wl.MIN_MINUTES_RATED} min): {career.is_rated.sum():,} of {len(career):,}")

player_career_stats: (3020, 87)
rated players (>= 45 min): 2,803 of 3,020


## Coverage by position
Position drives every percentile peer-group downstream, so we confirm the
rated population looks sane.

In [2]:
rated = career[career.is_rated]
summary = rated.groupby("position_code").agg(
    players=("player_id", "size"),
    median_minutes=("minutes_played", "median"),
    median_matches=("matches_played", "median"),
).reindex(wl.POSITIONS)
summary

,players,median_minutes,median_matches
position_code,,,
GK,227,1110.0,12.0
DF,987,1420.0,18.0
MD,1023,1194.0,20.0
FW,566,1122.5,22.0


## Accuracy rates — distribution sanity (Level 2 validation)
Known real-world ranges: keepers complete fewer passes, midfielders the most.

In [3]:
chk = rated.groupby("position_code")["pass_completion_pct"].median().reindex(wl.POSITIONS)
print("Median pass completion by position:")
print(chk.round(3).to_string())
assert 0.55 <= chk["GK"] <= 0.85, "GK pass completion out of expected range"
assert 0.78 <= chk["DF"] <= 0.92, "DF pass completion out of expected range"
assert 0.80 <= chk["MD"] <= 0.93, "MD pass completion out of expected range"
print("✓ pass completion by position within expected ranges")

fw_goals = rated[rated.position_code == "FW"]["goals_p90"]
print(f"\nFW goals per 90 — mean {fw_goals.mean():.3f}, max {fw_goals.max():.2f}")
assert fw_goals.max() < 3.0, "implausible FW goals_p90 — check minutes"
print("✓ FW goals per 90 distribution plausible")

Median pass completion by position:
position_code
GK    0.835
DF    0.830
MD    0.829
FW    0.760
✓ pass completion by position within expected ranges

FW goals per 90 — mean 0.284, max 1.67
✓ FW goals per 90 distribution plausible


## Goalkeeper save% — sanity

In [4]:
gk = rated[rated.position_code == "GK"].copy()
print(f"GK count: {len(gk)}  |  median save%: {gk.save_pct.median():.3f}  |  median saves/90: {gk.saves_p90.median():.2f}")
gk.nlargest(8, "minutes_played")[["short_name", "minutes_played", "saves", "conceded", "save_pct", "saves_p90"]].round(3)

GK count: 227  |  median save%: 0.768  |  median saves/90: 4.59


,short_name,minutes_played,saves,conceded,save_pct,saves_p90
246,H. Lloris,4440.0,164,45,0.785,3.324
163,T. Courtois,4230.0,180,45,0.800,3.830
256,J. Pickford,4110.0,214,65,0.767,4.686
280,David de Gea,4080.0,175,38,0.822,3.860
772,D. Subašić,3961.0,179,52,0.775,4.067
205,Ł. Fabiański,3930.0,229,58,0.798,5.244
1787,Alisson,3780.0,158,31,0.836,3.762
86,M. Ryan,3690.0,210,59,0.781,5.122


## What one player's career row looks like

In [5]:
kane = career[career.short_name == "H. Kane"].iloc[0]
cols = ["short_name", "position_code", "matches_played", "minutes_played",
        "goals", "goals_p90", "shots_total", "shot_accuracy_pct",
        "key_passes_p90", "pass_completion_pct", "competitions"]
career[career.short_name == "H. Kane"][cols].T

,242
short_name,H. Kane
position_code,FW
matches_played,47
minutes_played,3896.0
goals,35
goals_p90,0.808522
shots_total,183
shot_accuracy_pct,0.409836
key_passes_p90,0.254107
pass_completion_pct,0.751323


Phase 3 complete → the feature table is ready for a **football common-sense
review** before any rating is computed: **Phase 4**.